In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


In [9]:
# Load data
df1 = pd.read_csv("./data/bengaluru_house_prices.csv")
df1.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [10]:
# Rename 'total_sqft' to 'sqft' immediately (as requested)
df1 = df1.rename(columns={'total_sqft': 'sqft'})
print("Renamed columns:", df1.columns.tolist()) 


Renamed columns: ['area_type', 'availability', 'location', 'size', 'society', 'sqft', 'bath', 'balcony', 'price']


In [11]:
# Drop columns (your df2)
df2 = df1.drop(['area_type', 'society', 'balcony', 'availability'], axis='columns')
df2.shape

(13320, 5)

In [12]:
# Dropna (your df3)
df3 = df2.dropna()
df3.isnull().sum()

location    0
size        0
sqft        0
bath        0
price       0
dtype: int64

In [13]:
# Add 'bhk' (your code)
df3['bhk'] = df3['size'].apply(lambda x: int(x.split(' ')[0]))
df3.bhk.unique()

C:\Users\Mohd Uvais\AppData\Local\Temp\ipykernel_21956\1957629622.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['bhk'] = df3['size'].apply(lambda x: int(x.split(' ')[0]))


array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18])

In [14]:
# Functions (updated to use 'sqft')
def is_float(x):
    try:
        float(x)
    except:
        return False
    return True

In [15]:
df3[~df3['sqft'].apply(is_float)].head(10)

,location,size,sqft,bath,price,bhk
30,Yelahanka,4 BHK,2100 - 2850,4.0,186.000,4
122,Hebbal,4 BHK,3067 - 8156,4.0,477.000,4
137,8th Phase JP Nagar,2 BHK,1042 - 1105,2.0,54.005,2
165,Sarjapur,2 BHK,1145 - 1340,2.0,43.490,2
188,KR Puram,2 BHK,1015 - 1540,2.0,56.800,2
410,Kengeri,1 BHK,34.46Sq. Meter,1.0,18.500,1
549,Hennur Road,2 BHK,1195 - 1440,2.0,63.770,2
648,Arekere,9 Bedroom,4125Perch,9.0,265.000,9
661,Yelahanka,2 BHK,1120 - 1145,2.0,48.130,2
672,Bettahalsoor,4 Bedroom,3090 - 5002,4.0,445.000,4


In [16]:
def convert_sqft_to_num(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

In [17]:
# Apply conversion (your df4, updated for 'sqft')
df4 = df3.copy()
df4.sqft = df4.sqft.apply(convert_sqft_to_num)
df4 = df4[df4.sqft.notnull()]
df4.head(2)

,location,size,sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4


In [18]:
# Add price_per_sqft (your df5, updated for 'sqft')
df5 = df4.copy()
df5['price_per_sqft'] = df5['price'] * 100000 / df5['sqft']
df5.head()

,location,size,sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,2 BHK,1200.0,2.0,51.00,2,4250.000000


In [19]:
# Location cleaning (your code)
df5.location = df5.location.apply(lambda x: x.strip())
location_stats = df5['location'].value_counts(ascending=False)
location_stats
location_stats_less_than_10 = location_stats[location_stats <= 10]
location_stats_less_than_10
df5.location = df5.location.apply(lambda x: 'other' if x in location_stats_less_than_10 else x)
len(df5.location.unique())

241

In [20]:
# Sqft/BHK filter (your df6, updated for 'sqft')
df5[df5.sqft / df5.bhk < 300].head()
df6 = df5[~(df5.sqft / df5.bhk < 300)]
df6.shape

(12456, 7)

In [21]:
# Remove PPS outliers (your exact function: mean ± std per location)
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out


In [22]:
df7 = remove_pps_outliers(df6)
df7.shape

(10242, 7)

In [23]:
# Remove BHK outliers (your exact function)
def remove_bhk_outliers(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('bhk'):
            stats = bhk_stats.get(bhk - 1)
            if stats and stats['count'] > 5:
                exclude_indices = np.append(exclude_indices, bhk_df[bhk_df.price_per_sqft < (stats['mean'])].index.values)
    return df.drop(exclude_indices, axis='index')


In [24]:
df8 = remove_bhk_outliers(df7)
df8.shape

(7317, 7)

In [25]:
# Bath filter (your df9)
df8[df8.bath > df8.bhk + 2]
df9 = df8[df8.bath < df8.bhk + 2]
df9.shape

(7239, 7)

,location,sqft,bath,price,bhk
0,1st block jayanagar,2850.0,4.0,428.0,4
1,1st block jayanagar,1630.0,3.0,194.0,3
2,1st block jayanagar,1875.0,2.0,235.0,3


In [ ]:
df10 = df9.drop(['size', 'price_per_sqft'], axis='columns')
df10.location=df10.location.str.lower()
df10.head(3)

In [40]:
df10.to_csv('data/processed_data.csv', index=False)

In [27]:
# === PIPELINE VERSION (Automated OHE - No Manual Dummies) ===
# Prepare raw X/y (from df10: keep 'location' as string for pipeline)
X = df10.drop('price', axis=1)  # Raw: ['location', 'sqft', 'bath', 'bhk']
y = df10['price']
print("Raw X shape for pipeline:", X.shape)
print("Raw X columns:", X.columns.tolist())
X.head(3)
y.head(3)

Raw X shape for pipeline: (7239, 4)
Raw X columns: ['location', 'sqft', 'bath', 'bhk']


0    428.0
1    194.0
2    235.0
Name: price, dtype: float64

In [28]:
# Build ColumnTransformer (Automates OHE on 'location' + passthrough numerical)
# OneHotEncoder: Converts strings to dummies (incl. 'other'); drop='first' mimics your drop('other')
ohe = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)  # No multicollinearity
column_trans = make_column_transformer(
    (ohe, ['location']),  # Auto: String 'location' → one-hot dummies (drops one category)
    remainder='passthrough'  # Auto: Keep 'sqft', 'bath', 'bhk' unchanged
)


In [29]:
# Build Pipeline (Preprocess + Model - All in One Object)
pipe = Pipeline([
    ('preprocessor', column_trans),  # Transforms raw X to dummies + numerical
    ('model', LinearRegression())    # Fits on transformed data
])


In [30]:
print("\n=== Pipeline Ready: Raw Input → Auto OHE → Passthrough → LinearRegression ===")



=== Pipeline Ready: Raw Input → Auto OHE → Passthrough → LinearRegression ===


In [31]:
# Train-test split (your exact code)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)


In [32]:
# Fit pipeline (ONE CALL: Handles OHE + model fitting)
pipe.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
# Single split score (your code)
print("Pipeline R² (single split):", pipe.score(X_test, y_test))  # Expect ~0.75-0.85


Pipeline R² (single split): 0.8629132245229317


In [34]:
# Cross-validation (your exact code - works directly on pipeline)
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)
cv_scores = cross_val_score(pipe, X, y, cv=cv)  # ONE CALL: Splits + transforms + scores each fold
print("Pipeline CV Scores:", cv_scores)  # Expect array like [0.82-0.86]
print("Pipeline CV Mean:", cv_scores.mean(), "±", cv_scores.std())


Pipeline CV Scores: [0.82702546 0.86027005 0.85322178 0.8436466  0.85481502]
Pipeline CV Mean: 0.8477957812446905 ± 0.011687089323360688


In [35]:
def predict_price(location, sqft, bath, bhk):
    input_data = pd.DataFrame({
        'location': [location.lower().strip()],  # Raw string - pipeline auto-encodes
        'sqft': [sqft],
        'bath': [bath],
        'bhk': [bhk]
    })
    pred = pipe.predict(input_data)[0]
    return round(float(pred), 1)



In [ ]:
predict_price('1st Phase JP Nagar', 1000, 2, 2)

83.9

In [37]:
predict_price('Indira Nagar', 1000, 2, 2)

193.3

In [38]:
import pickle
with open('home_prices.pkl', 'wb') as f:
    pickle.dump(pipe, f)
print("\n✅ Pipeline saved as home_prices.pkl (load in FastAPI)")


✅ Pipeline saved as home_prices.pkl (load in FastAPI)


In [147]:

predict_price('1st Phase JP Nagar',1000, 3, 3)

86.1

In [148]:

predict_price('Indira Nagar',1000, 3, 3)

195.5

In [149]:
# Get clean feature names (for JSON export)
feature_names = pipe.named_steps['preprocessor'].get_feature_names_out()
clean_feature_names = [name.split('__')[-1] for name in feature_names]  # e.g., 'location_whitefield' -> 'whitefield'


In [150]:
# Save columns.json
columns = {'data_columns': clean_feature_names}
with open("columns.json", 'w') as f:
    json.dump(columns, f, indent=2)
print("Clean feature names:")
# print(clean_feature_names)


Clean feature names:
['location_1st phase jp nagar', 'location_2nd phase judicial layout', 'location_2nd stage nagarbhavi', 'location_5th block hbr layout', 'location_5th phase jp nagar', 'location_6th phase jp nagar', 'location_7th phase jp nagar', 'location_8th phase jp nagar', 'location_9th phase jp nagar', 'location_abbigere', 'location_aecs layout', 'location_akshaya nagar', 'location_ambalipura', 'location_ambedkar nagar', 'location_amruthahalli', 'location_anandapura', 'location_ananth nagar', 'location_anekal', 'location_anjanapura', 'location_ardendale', 'location_arekere', 'location_attibele', 'location_babusapalaya', 'location_badavala nagar', 'location_balagere', 'location_banashankari', 'location_banashankari stage ii', 'location_banashankari stage iii', 'location_banashankari stage v', 'location_banashankari stage vi', 'location_banaswadi', 'location_banjara layout', 'location_bannerghatta', 'location_bannerghatta road', 'location_basavangudi', 'location_basaveshwara naga